[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-anomaly.ipynb)

# Anomaly & Outlier Detection

*AIBits Academy · Machine Learning End To End · Unsupervised Learning · New*

DBSCAN flags noise as a side effect of clustering. This chapter covers algorithms purpose-built to find rare, unusual points as the primary goal — fraud, defects, and intrusions.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Why Not Just Use Classification?

Fraud, equipment failure, and network intrusions share a structural problem: the "positive" class is extremely rare (often <0.1%) and its exact form keeps changing (fraudsters adapt). Anomaly detection algorithms are built to learn "what normal looks like" from mostly-normal data, then flag anything that deviates — **without needing labelled examples of every possible anomaly type**, which classification requires.

## Isolation Forest

The key insight: anomalies are "few and different" — they should be *easier to isolate* with random partitioning than normal points, which sit in dense regions requiring many splits to separate from their neighbours.

> **🎨 The Visual Analogy — Splitting the Crowd**
>
> Picture a crowded room where most people are packed tightly in the centre (the **inliers**), while one person stands alone in a far corner (the **outlier**). Now start drawing random straight lines to slice the room into smaller and smaller zones until each person is alone in their own space. The lone person in the corner? One or two cuts and they're isolated — there's nobody near them. Anyone in the tight central crowd? You'll have to slice line after line after line to fence off a single person from all their close neighbours. Isolation Forest turns exactly that asymmetry into an anomaly score: **the number of cuts needed to isolate a point** (its path length in the tree) is short for outliers and long for inliers.

- Build many random trees: at each node, pick a random feature and a random split value between its min and max

- Recursively partition until every point is isolated in its own leaf

- An anomaly, being far from the dense mass of normal points, gets isolated in very **few splits** (short path length from root to leaf)

- A normal point, embedded deep in a dense cluster, requires **many splits** to isolate

## Splitting the Crowd — Watch Isolation Depth Diverge

The two rooms below run genuine random axis-aligned cuts (a real isolation tree, replayed step-by-step) on the same crowd. Left isolates the lone corner outlier; right isolates a point buried in the central crowd. Same algorithm, same data — only the target differs.

$$s(x) = 2^{-E[h(x)]/c(n)} \qquad \text{where } h(x) = \text{path length},\ c(n) = \text{average path length normaliser}$$

Score close to 1 → likely anomaly (short average path). Score close to 0.5 → likely normal (average path length, no isolation advantage).

## Code — Isolation Forest for HDFC Transaction Fraud

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

np.random.seed(8)
# 2000 normal transactions + 20 genuinely anomalous ones (unusual amount/hour combos)
normal = np.column_stack([np.random.normal(1500,600,2000).clip(50,None),
                          np.random.normal(14,4,2000).clip(0,23)])
anomalies = np.column_stack([np.random.uniform(40000,90000,20),
                             np.random.uniform(1,4,20)])  # huge amount, unusual hour
X = np.vstack([normal, anomalies])
true_labels = np.array([1]*2000 + [-1]*20)  # 1=normal, -1=anomaly (sklearn convention)

iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=42)
pred = iso.fit_predict(X)
scores = iso.decision_function(X)  # higher = more normal

caught = ((pred == -1) & (true_labels == -1)).sum()
false_alarms = ((pred == -1) & (true_labels == 1)).sum()
print(f"Anomalies caught: {caught}/20")
print(f"False alarms on normal transactions: {false_alarms}")
print(f"\nMost anomalous transaction: amount=₹{X[np.argmin(scores)][0]:.0f}, hour={X[np.argmin(scores)][1]:.0f}")

## One-Class SVM

Learns a boundary (using the RBF kernel machinery from the SVM/Kernel Methods chapters) that encloses the dense "normal" region as tightly as possible, then flags anything outside it. Effective in high dimensions but more sensitive to the `nu` and `gamma` hyperparameters than Isolation Forest, and O(n²)-ish training cost limits it to smaller datasets.

In [ ]:
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

Xs = StandardScaler().fit_transform(X)
ocsvm = OneClassSVM(kernel='rbf', nu=0.01, gamma='scale')
pred_svm = ocsvm.fit_predict(Xs)
print(f"One-Class SVM anomalies caught: {((pred_svm==-1)&(true_labels==-1)).sum()}/20")

## Local Outlier Factor (LOF)

Unlike Isolation Forest and One-Class SVM (which learn one global notion of "normal"), LOF measures each point's density **relative to its local neighbourhood** — essential when normal density genuinely varies across the dataset (exactly the varying-density problem raised for DBSCAN):

$$\mathrm{LOF}(x) = \dfrac{\text{average local density of } x\text{'s } k\text{-neighbours}}{\text{local density of } x}$$

LOF ≈ 1 → similar density to neighbours (normal). LOF ≫ 1 → x sits in a much sparser region than its neighbours (local outlier) — even if that region's absolute density would look "normal" elsewhere in the dataset.

## EllipticEnvelope — When "Normal" Is Genuinely Gaussian

If the normal class genuinely follows a roughly Gaussian distribution (common for well-behaved sensor readings or physical measurements), `EllipticEnvelope` fits a robust estimate of the data's mean and covariance, then flags points with an unusually large **Mahalanobis distance** from that centre — effectively drawing an ellipse (or ellipsoid, in higher dimensions) around the normal region:

$$D^2_{\text{Mahalanobis}}(x) = (x-\mu)^\mathsf{T} \Sigma^{-1} (x-\mu) \qquad \text{— accounts for feature correlation, unlike raw Euclidean distance}$$

In [ ]:
from sklearn.covariance import EllipticEnvelope

# Best suited to the HDFC transaction data from earlier — assumes roughly elliptical "normal" region
ee = EllipticEnvelope(contamination=0.01, random_state=42)
pred_ee = ee.fit_predict(X)
print(f"EllipticEnvelope anomalies caught: {((pred_ee==-1)&(true_labels==-1)).sum()}/20")
# Uses a robust covariance estimator (Minimum Covariance Determinant) so a handful of
# outliers already in the training data don't themselves distort the fitted "normal" ellipse

> **⚠ The Gaussian Assumption Is Load-Bearing**
>
> EllipticEnvelope is fast and statistically elegant, but it assumes the normal class is unimodal and roughly elliptical. If normal behaviour actually forms several separate clusters (e.g., "normal" daytime spending and "normal" nighttime spending look different but are each individually fine), a single ellipse will either miss real anomalies between the clusters or falsely flag the gap between legitimate clusters as anomalous. Isolation Forest or LOF make no such distributional assumption and handle multi-modal "normal" far better.

## Choosing an Algorithm

| Algorithm | Best for | Watch out for |
|---|---|---|
| Isolation Forest | Large datasets, high dimensions, fast, minimal tuning | Less effective when anomalies form dense sub-clusters rather than being scattered |
| One-Class SVM | Smaller datasets, when a tight, well-defined normal boundary is expected | O(n²)-ish cost; sensitive to kernel/nu tuning |
| Local Outlier Factor | Data with genuinely varying local density (matches HDBSCAN's use case) | No easy way to score new unseen points without refitting (unless using `novelty=True`) |
| EllipticEnvelope | Normal class is genuinely unimodal & roughly Gaussian; fast, interpretable | Breaks down badly on multi-modal or non-elliptical "normal" regions |
| DBSCAN noise flag | Already clustering anyway and outliers are incidental | Not purpose-built — global ε struggles with the same varying-density issue as LOF solves |

> **🔗 Real-World Case Study — Predictive Maintenance on the NASA IMS Bearing Dataset**
>
> A widely-cited industrial application (Flovik, 2019) applies exactly the PCA + Mahalanobis-distance combination above to **predictive maintenance**: detecting a mechanical bearing failure days before it happens, using only vibration-sensor readings from four bearings sampled every 10 minutes.
>
> **Method:** PCA compresses the 4 correlated vibration-sensor channels down to their top 2 principal components, fit only on data from a known-healthy operating period. The Mahalanobis distance of each new reading to that healthy-period centre is then tracked over time, with the anomaly threshold set at 3 standard deviations above the mean distance observed during the healthy period (empirically, **threshold ≈ 3.8** in the original study).
>
> **Result:** the Mahalanobis distance stayed flat and low for days, then began climbing steadily roughly **3 days before the actual bearing failure** — well before the failure would have been apparent from raw vibration readings alone — crossing the anomaly threshold with enough lead time to schedule maintenance instead of reacting to an unplanned breakdown.
>
> > **⚠ Advanced Topic — Neural Networks & Deep Learning Course**
> >
> > The original study also fits an **autoencoder neural network** as a second approach to the same problem — compressing the sensor readings through a bottleneck layer and using reconstruction error (instead of Mahalanobis distance) as the anomaly signal. It reaches a similar conclusion (failure detected ~3 days early) via a different mechanism. Autoencoders are out of scope for this course (neural networks generally are) and are logged for a future dedicated Neural Networks / Deep Learning course — the PCA + Mahalanobis method above is a fully complete, self-contained technique on its own and doesn't require the neural-network half to be useful in practice.

## Try It — Isolation Depth, Live

Below is a real 60-transaction sample (55 normal + 5 genuine anomalies) drawn from the exact same distributions as the code above. Click any point — or anywhere else on the plot — to run genuine random-partition isolation live: 150 fresh random isolation trees are built in your browser right now, the average path length to isolate that exact point is measured, and it's converted into the real anomaly-score formula from the page, s(x)=2^(−E[h(x)]/c(n)). Notice how few splits it takes to isolate the ₹40,000+ outliers compared to the dense cluster of normal transactions.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · The z-score rule

Write `zscore_outliers(x, thr=3)` returning the **indices** of values whose absolute z-score exceeds `thr`.

In [ ]:
import numpy as np
def zscore_outliers(x, thr=3):
    pass   # TODO


In [ ]:
try:
    x = np.concatenate([np.random.default_rng(0).normal(50, 2, 200), [95]])
    idx = zscore_outliers(x)
    check("flags the planted value", 200 in list(idx))
    check("flags very few", len(idx) <= 3)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def zscore_outliers(x, thr=3):
    z = (x - x.mean()) / x.std()
    return np.where(np.abs(z) > thr)[0]

```

</details>

### Exercise 2 · Medium · Isolation Forest

Fit `IsolationForest(contamination=0.02, random_state=0)` on `X` and store the labels of the two probe points in `probe_labels` (`1` = normal, `-1` = anomaly).

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
rng = np.random.default_rng(1)
X = rng.normal(0, 1, (500, 2))
probe = np.array([[0.1, -0.2], [7.0, 7.0]])
probe_labels = None   # TODO


In [ ]:
try:
    check("centre is normal, far point is anomalous", list(probe_labels) == [1, -1])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.ensemble import IsolationForest
rng = np.random.default_rng(1)
X = rng.normal(0, 1, (500, 2))
probe = np.array([[0.1, -0.2], [7.0, 7.0]])
probe_labels = IsolationForest(contamination=0.02, random_state=0).fit(X).predict(probe)

```

</details>

### Exercise 3 · Stretch · Precision and recall of a detector

The data contains 10 known planted anomalies (`truth == -1`). Fit an Isolation Forest (`contamination=0.02`), then compute the **recall** (share of planted anomalies caught) and the **precision** (share of flagged points that are truly anomalous). Store them in `recall` and `precision`.

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
rng = np.random.default_rng(2)
X = np.vstack([rng.normal(0, 1, (490, 2)), rng.uniform(5, 8, (10, 2)) * rng.choice([-1, 1], (10, 2))])
truth = np.array([1] * 490 + [-1] * 10)
recall = precision = None   # TODO


In [ ]:
try:
    check("catches most planted anomalies", recall >= 0.8)
    check("precision is meaningful", 0.3 <= precision <= 1.0)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.ensemble import IsolationForest
rng = np.random.default_rng(2)
X = np.vstack([rng.normal(0, 1, (490, 2)), rng.uniform(5, 8, (10, 2)) * rng.choice([-1, 1], (10, 2))])
truth = np.array([1] * 490 + [-1] * 10)
pred = IsolationForest(contamination=0.02, random_state=0).fit_predict(X)
tp = ((pred == -1) & (truth == -1)).sum()
recall = tp / (truth == -1).sum()
precision = tp / max((pred == -1).sum(), 1)

```

`contamination` is your belief about the anomaly rate; setting it too high buys recall at the cost of precision.

</details>

---
*Back to the course: **Machine Learning End To End → Anomaly & Outlier Detection**.*